# 05 — Services

ROS 2 Services enable **request-response communication** between nodes. A client sends a request to a service server, which processes it and returns a response.

Unlike Topics, which are designed for continuous data exchange, Services are suitable for discrete operations that need a response.

## 1. Service Communication

A service interaction involves three main elements:

- **Service Client:** Sends a request and receives the response.
- **Service Server:** Receives the request, executes the operation, and returns the response.
- **Service Interface:** Defines the request and response data structures.

Example: A client requests the sum of two integers. The server calculates the result and returns it.

```text
Service Client
     |
     | Request: a = 5, b = 3
     v
Service Server
     |
     | Response: sum = 8
     v
Service Client
```

## 2. Topics vs. Services

| Feature | Topics | Services |
|---|---|---|
| Communication | Publish-subscribe | Request-response |
| Data flow | Continuous or event-driven streams | Individual requests and responses |
| Communication roles | Publisher and subscriber | Client and server |
| Typical use | Sensor data, odometry, velocity commands | Calculations, queries, discrete operations |

Services are not intended for long-running operations that require continuous feedback. ROS 2 Actions are generally better suited to those tasks.

## 3. Service Interfaces

A service interface defines the structure of its request and response.

ROS 2 provides the example service type `example_interfaces/srv/AddTwoInts`.

Inspect its definition:

```bash
ros2 interface show example_interfaces/srv/AddTwoInts
```

Expected interface:

```text
int64 a
int64 b
---
int64 sum
```

The fields above `---` define the request. The field below it defines the response.

- Request: `a` and `b`
- Response: `sum`

If the interface package is unavailable, install it on Ubuntu with ROS 2 Humble:

```bash
sudo apt install ros-humble-example-interfaces
```

## 4. Discover Services

List currently available services:

```bash
ros2 service list
```

Inspect the type of a service:

```bash
ros2 service type /add_two_ints
```

List services together with their types:

```bash
ros2 service list -t
```

Show the request and response interface:

```bash
ros2 interface show example_interfaces/srv/AddTwoInts
```

Service names depend on which nodes are running.

## 5. Python Service Server and Client

The practical exercises in this section use Python and `rclpy`.

### Service Server

The server creates a service and registers a callback to process incoming requests.

Core pattern:

```python
from example_interfaces.srv import AddTwoInts

self.service = self.create_service(
    AddTwoInts,
    'add_two_ints',
    self.add_callback
)
```

The callback receives the request and a response object:

```python
def add_callback(self, request, response):
    response.sum = request.a + request.b
    return response
```

The service name is `add_two_ints`. The callback calculates the sum and returns the response.

### Service Client

The client creates a client for the same service type and name:

```python
from example_interfaces.srv import AddTwoInts

self.client = self.create_client(
    AddTwoInts,
    'add_two_ints'
)
```

A request can be created and populated:

```python
request = AddTwoInts.Request()
request.a = 5
request.b = 3
```

The client sends the request asynchronously:

```python
future = self.client.call_async(request)
```

The future represents the pending response. The node must continue processing ROS events so the response can be received.

The complete server and client implementations are developed in the examples directory.

## 6. Test a Service from the Terminal

Once the server is running, list the available services:

```bash
ros2 service list
```

Inspect the service type:

```bash
ros2 service type /add_two_ints
```

Call the service with a request:

```bash
ros2 service call /add_two_ints example_interfaces/srv/AddTwoInts "{a: 5, b: 3}"
```

Expected response:

```text
sum: 8
```

The service server must be running and available before the call can succeed.

## 7. Learning Objectives

After completing this section, you should be able to:

- Explain the difference between Topics and Services.
- Identify service clients, servers, names, and types.
- Inspect service interfaces using the ROS 2 CLI.
- Implement a Python service server using `rclpy`.
- Implement a Python client that sends requests and receives responses.
- Test a service using `ros2 service call`.

## 8. Exercises

1. Create a Python service server that adds two integers.
2. Create a separate client that requests the sum of two numbers.
3. Test the service from the command line.
4. Modify the server to subtract two integers.
5. Test what happens when the client requests a service that is not available.

**Key takeaway:** ROS 2 Services provide structured request-response communication between nodes. The service interface defines the request and response, the server processes requests, and the client initiates the interaction.